In [ ]:
# app.py - Flask Backend
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
import json
import os
import requests
from bs4 import BeautifulSoup
import re
import trafilatura
import openai
from typing import Dict, List
import tldextract
import html2text
from urllib.parse import urljoin, urlparse
import time
import random

In [ ]:
app = Flask(__name__)
CORS(app)

# Configuration
app.config['SECRET_KEY'] = os.environ.get('SECRET_KEY', 'your-secret-key-change-in-production')
app.config['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY')

# In-memory storage (use database in production)
enriched_companies = {}

class WebScraper:
    """Production scraper for web application"""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })
    
    def clean_html(self, html_content: str) -> str:
        """Clean HTML content to extract meaningful text"""
        soup = BeautifulSoup(html_content, 'lxml')
        
        # Remove unwanted elements
        for element in soup.find_all(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            element.decompose()
        
        # Convert to text
        h = html2text.HTML2Text()
        h.ignore_links = False
        h.ignore_images = True
        text = h.handle(str(soup))
        
        # Clean up
        text = re.sub(r'\n\s*\n', '\n\n', text)
        return text.strip()
    
    def extract_contact(self, text: str) -> Dict:
        """Extract emails and phones"""
        # Email extraction
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        emails = list(set(re.findall(email_pattern, text)))
        
        # Filter invalid emails
        invalid = ['example', 'test', 'user', 'email', '.png', '.jpg']
        emails = [e for e in emails if not any(i in e.lower() for i in invalid)][:3]
        
        # Phone extraction
        phone_patterns = [
            r'\+\d{1,3}[\s\-]?\d{1,4}[\s\-]?\d{1,4}[\s\-]?\d{1,4}',
            r'\d{3}[\s\-]?\d{3}[\s\-]?\d{4}',
            r'\(\d{3}\)\s?\d{3}[\s\-]?\d{4}',
        ]
        
        phones = set()
        for pattern in phone_patterns:
            phones.update(re.findall(pattern, text))
        
        return {
            'emails': list(phones)[:3] if phones else [],
            'phones': list(phones)[:3] if phones else []
        }
    
    def scrape(self, url: str) -> Dict:
        """Scrape company website"""
        try:
            # Try trafilatura first
            downloaded = trafilatura.fetch_url(url)
            if downloaded:
                content = trafilatura.extract(downloaded, favor_precision=True)
                cleaned = self.clean_html(downloaded)
                contact = self.extract_contact(cleaned)
                
                return {
                    'success': True,
                    'content': cleaned[:5000],  # Limit content size
                    'contact': contact,
                    'title': BeautifulSoup(downloaded, 'lxml').title.string if BeautifulSoup(downloaded, 'lxml').title else ''
                }
            
            # Fallback to requests
            response = self.session.get(url, timeout=15, verify=False)
            if response.status_code == 200:
                cleaned = self.clean_html(response.text)
                contact = self.extract_contact(cleaned)
                soup = BeautifulSoup(response.text, 'lxml')
                
                return {
                    'success': True,
                    'content': cleaned[:5000],
                    'contact': contact,
                    'title': soup.title.string if soup.title else ''
                }
            
        except Exception as e:
            pass
        
        return {'success': False, 'error': 'Failed to scrape website'}

class AIAnalyzer:
    """AI-powered company analysis"""
    
    def __init__(self):
        self.client = openai.OpenAI(api_key=app.config['OPENAI_API_KEY'])
    
    def analyze(self, scraped_data: Dict, url: str, company_name: str) -> Dict:
        """Analyze company data"""
        if not scraped_data.get('success'):
            return self.empty_profile(url, company_name)
        
        prompt = f"""Based ONLY on this company website content, extract information.
DO NOT fabricate any data. Use "N/A" if not found.

Content: {scraped_data['content'][:3000]}

Return JSON:
{{
  "company_name": "{company_name or 'N/A'}",
  "description": "Brief description",
  "industry": "Industry type or N/A",
  "services": ["Service 1"],
  "emails": {json.dumps(scraped_data['contact']['emails'])},
  "phone_numbers": {json.dumps(scraped_data['contact']['phones'])},
  "address": "Address or N/A"
}}"""
        
        try:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo-1106",
                messages=[
                    {"role": "system", "content": "Extract ONLY explicitly mentioned information. Never fabricate data."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            
            return json.loads(response.choices[0].message.content)
        except:
            return self.empty_profile(url, company_name)
    
    def empty_profile(self, url: str, name: str) -> Dict:
        return {
            'company_name': name or 'N/A',
            'website': url,
            'description': 'N/A',
            'industry': 'N/A',
            'services': [],
            'emails': [],
            'phone_numbers': [],
            'address': 'N/A'
        }

In [ ]:
# Initialize
scraper = WebScraper()
analyzer = AIAnalyzer()

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/enrich', methods=['POST'])
def enrich():
    """Enrich endpoint - POST /enrich"""
    data = request.json
    url = data.get('url', '').strip()
    company_name = data.get('company_name', '').strip()
    
    if not url:
        return jsonify({'error': 'URL is required'}), 400
    
    # Add http if missing
    if not url.startswith('http'):
        url = 'https://' + url
    
    # Scrape
    scraped_data = scraper.scrape(url)
    
    # Analyze
    profile = analyzer.analyze(scraped_data, url, company_name)
    
    # Store
    enriched_companies[url] = profile
    
    return jsonify(profile)

@app.route('/results', methods=['GET'])
def results():
    """Get all results - GET /results"""
    return jsonify(list(enriched_companies.values()))

if __name__ == '__main__':
    # Disable SSL warnings for scraping
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    app.run(debug=True, host='0.0.0.0', port=int(os.environ.get('PORT', 5000)))